# RoomWalk: video → 3D room (Gaussian splat)

Turns a slow walkthrough video of a room into a `splat.ply` you can walk through in the RoomWalk viewer (WASD + mouse).

**Before you run:** Runtime → Change runtime type → **GPU** (T4 is fine). Then run the cells top to bottom.

Timing on a free T4: install ~6–10 min · COLMAP ~5–15 min · training ~15–30 min.

Pipeline: `ffmpeg` frames → `COLMAP` camera poses → `nerfstudio splatfacto` training → `ns-export gaussian-splat`.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)

In [ ]:
#@title 2. Install COLMAP + nerfstudio (6–10 min)
import torch, re, subprocess, sys
%cd /content
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
# gsplat ships prebuilt CUDA wheels per torch/cuda version; use them when they exist, else build from source.
tv = ''.join(torch.__version__.split('+')[0].split('.')[:2])          # e.g. 2.5.1 -> 25
cv = ''.join(torch.version.cuda.split('.')[:2])                       # e.g. 12.4 -> 124
idx = f'https://docs.gsplat.studio/whl/pt{tv}cu{cv}'
print('trying gsplat wheels from', idx)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat', '--index-url', idx])
if r.returncode != 0:
    print('no prebuilt wheel; building gsplat from source (takes longer)')
    !pip install -q gsplat
!pip install -q nerfstudio
!ns-train --help > /dev/null && echo 'nerfstudio OK'
!colmap -h | head -1

In [ ]:
#@title 3. Upload your room video
#@markdown Film slowly, keep the room lit, overlap a lot, and avoid mirrors and windows. 60–120 seconds is plenty.
from google.colab import files
import shutil, os
os.makedirs('/content/input', exist_ok=True)
up = files.upload()
video_path = None
for name in up:
    dst = f'/content/input/{name}'
    shutil.move(name, dst)
    video_path = dst
print('video:', video_path)

In [ ]:
#@title 4. Extract frames and solve camera poses with COLMAP (5–15 min)
num_frames = 200  #@param {type:"integer"}
#@markdown 150–300 frames is the sweet spot. More frames = better model but slower COLMAP.
!rm -rf /content/data/room
!ns-process-data video --data "$video_path" --output-dir /content/data/room \
    --num-frames-target $num_frames --no-gpu --matching-method sequential --verbose
!echo; echo 'Registered images:'; ls /content/data/room/images | wc -l

If COLMAP registered well under half the frames, the video is the problem: too fast, too dark, or too many plain surfaces. Re-film slower, with more overlap, and keep a few objects in frame at all times.

In [ ]:
#@title 5. Train the Gaussian splat (15–30 min)
iterations = 15000  #@param {type:"integer"}
#@markdown 7000 = quick preview (~10 min). 15000 = good. 30000 = best.
!ns-train splatfacto --data /content/data/room --output-dir /content/outputs \
    --experiment-name room --vis tensorboard \
    --max-num-iterations $iterations \
    --pipeline.model.cull-alpha-thresh 0.005 \
    --pipeline.model.use-scale-regularization True

In [ ]:
#@title 6. Export splat.ply and download it
import glob, os
cfg = sorted(glob.glob('/content/outputs/room/splatfacto/*/config.yml'))[-1]
print('config:', cfg)
!rm -rf /content/exports/room
!ns-export gaussian-splat --load-config "$cfg" --output-dir /content/exports/room
ply = '/content/exports/room/splat.ply'
print('size: %.1f MB' % (os.path.getsize(ply) / 1e6))
from google.colab import files
files.download(ply)

## Walk through it

1. In the `roomwalk/viewer` folder run `python3 -m http.server 8000` and open http://localhost:8000
2. Click **Open scan…** and pick `splat.ply` (or drag it onto the page).
3. Click the room, then WASD to walk and the mouse to look. If the room looks tilted or upside down, press **U** to cycle the up axis or **F** to flip it. **[** and **]** change walking speed.

**Optional cleanup:** floaters (blurry blobs in the air) are normal. The viewer ignores the outer 2% of points when it places you, and `--pipeline.model.cull-alpha-thresh` above removes most faint ones. For a cleaner model, film more overlap or train longer.